# 🦗 Gradient ascent: a grasshopper climbs a hill

A grasshopper stands on a hill and wants to reach the top. From where it stands it feels only
the ground under its feet, so it has to decide where to jump from that.

Its whole plan fits in two lines of pseudocode:

```
while not flat:
    jump in the direction that increases height
```

To follow this plan, the grasshopper has to know which way is up and how steep the ground is.
One number tells it both: the **gradient**. The grasshopper measures it by trying a tiny step:

1. Pick a small step size, `eps`.
2. Ask how much the altitude changes if the grasshopper moves `eps` to the right. That change
   is `delta`.
3. The gradient is the ratio between the change in altitude and the step size:
   `grad = delta / eps`.

A positive gradient points to the right, a negative gradient points to the left, and the larger
its size, the steeper the hill. On flat ground the gradient is close to zero, so "not flat"
means that the gradient is not close to zero.

In this notebook you write the two functions of the recap:

- `grad(f, x)` measures the gradient of the hill `f` at the place `x`.
- `gradient_ascent(f, x, alpha=0.1)` is the whole climb. It starts at `x`, jumps until the
  ground is flat, and returns where the grasshopper ended up.

Everywhere you see 🎯 it is your turn. There are five 🎯 lines, two in Piece 1 and three in
Piece 2. After each piece, a check cell tells you whether it works. All the other code is
written for you.

Run the cells from top to bottom. The grey bar called Setup only prepares the pictures and the
checks: run it and move on. When a check says Not yet, read what it says, fix your lines and run
the piece and its check again. A red box is normal too: read its last line.

In [ ]:
#@title ⚙️ Setup: imports and helpers (nothing to fill in here) { display-mode: "form" }
import ast
import inspect
import textwrap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Circle, Ellipse, PathPatch
from matplotlib.path import Path
from matplotlib.transforms import Affine2D, ScaledTranslation

# One colour, one meaning. The palette is the one of the grasshopper game.
HOPPER, HOPPER_DARK = "#3f7d5f", "#2c5c45"        # the grasshopper
GROUND, SKY, INK = "#e4e0d3", "#fdfcf7", "#6b6b6b"  # the hill, the sky, the axes
TRAIL, TRAIL_FAINT = "#7a5c8c", "#c3b3ce"         # the places where it has stood
PEAK = "#c08a3e"                                  # the top of the hill
X_MIN, X_MAX = 0.0, 6.0                           # the stretch of ground in the pictures
X_TOP = 3.0                                       # where the top of the hill is
PIECE_DONE = {"grad": False, "ascent": False}
plt.rcParams["figure.dpi"] = 100                  # the same picture size in every notebook viewer


# ==== drawing ================================================================

def draw_hopper(ax, x, y, size=0.8, face=1, tilt=0.0):
    """The grasshopper of the game, `size` inches long, standing at the point (x, y)."""
    k = size / 34.0                                     # glyph units to inches
    pt = 72 * k                                         # one glyph unit in points
    place = (Affine2D().scale(face * k, -k).rotate_deg(tilt)
             + ax.figure.dpi_scale_trans + ScaledTranslation(x, y, ax.transData))

    def stroke(points, curve, width, colour, alpha=1.0):
        codes = [Path.MOVETO] + [Path.CURVE3 if curve else Path.LINETO] * (len(points) - 1)
        return PathPatch(Path(points, codes), fc="none", ec=colour, lw=width * pt,
                         capstyle="round", joinstyle="round", alpha=alpha)

    parts = [
        Ellipse((-1, -8.5), 26.6, 13.8, fc="none", ec=SKY, lw=2.6 * pt),   # a light halo
        stroke([(-2, -8), (-13, -13), (-16.5, -0.5)], False, 1.8, HOPPER_DARK),   # hind leg
        stroke([(3.5, -6.5), (6.5, -0.5)], False, 1.8, HOPPER_DARK),              # front leg
        Ellipse((-1, -8.5), 24, 11.2, fc=HOPPER, ec="none"),                      # body
        stroke([(-10, -10.6), (-2, -13.8), (6, -10.2)], True, 1.1, HOPPER_DARK, 0.7),  # wing
        Circle((10.6, -11), 4.8, fc=HOPPER, ec="none"),                           # head
        Circle((12.3, -11.8), 1.5, fc=HOPPER_DARK, ec="none"),                    # eye
        stroke([(13.6, -14), (18.6, -18.6), (21.6, -21.6)], True, 1.0, HOPPER_DARK),   # antenna
    ]
    for z, part in enumerate(parts):
        part.set_transform(place)
        part.set_clip_on(False)
        part.set_zorder(8 + z * 0.01)
        ax.add_artist(part)


def slope_tilt(ax, f, x, lean=0.65):
    """How far the grasshopper leans (in degrees) to follow the hill at x, as drawn on the page."""
    width, height = ax.figure.get_size_inches() * np.array(ax.get_position().size)
    (left, right), (bottom, top) = ax.get_xlim(), ax.get_ylim()
    d = 1e-4
    slope = (f(x + d) - f(x - d)) / (2 * d)
    on_page = slope * (height / (top - bottom)) / (width / (right - left))
    return lean * float(np.degrees(np.arctan(on_page)))


def hill_axes(f, title):
    fig, ax = plt.subplots(figsize=(7.2, 3.5))
    fig.subplots_adjust(left=0.085, right=0.985, bottom=0.17, top=0.88)
    xs = np.linspace(X_MIN, X_MAX, 300)
    ys = np.array([f(v) for v in xs])
    ax.set_facecolor(SKY)
    ax.fill_between(xs, ys, 0, color=GROUND, zorder=1)
    ax.plot(xs, ys, color=INK, lw=2, zorder=2)
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(0, 1.32 * ys.max())                     # headroom for the grasshopper
    ax.set_xlabel("place x", color=INK)
    ax.set_ylabel("altitude f(x)", color=INK)
    ax.set_title(title, loc="left", fontsize=11, color=INK)
    ax.tick_params(colors=INK)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(INK)
    ax.plot([X_TOP, X_TOP], [0, f(X_TOP)], color=PEAK, lw=1, linestyle=":", zorder=2)
    ax.plot([X_TOP], [f(X_TOP)], marker="*", ms=14, color=PEAK, zorder=3, linestyle="none")
    ax.text(X_TOP + 0.08, 0.3, "top of the hill: x = %g" % X_TOP, color=PEAK, fontsize=10)
    return fig, ax


def draw_hill(f, x0):
    """The hill, with the grasshopper standing at the place x0."""
    fig, ax = hill_axes(f, "the hill f and the grasshopper at x0 = %g" % x0)
    ax.plot([x0, x0], [0, f(x0)], color=INK, lw=1, linestyle=":", zorder=2)
    ax.text(x0, -0.9, "x0", ha="center", va="top", color=INK, fontsize=10, clip_on=False)
    draw_hopper(ax, x0, f(x0), tilt=slope_tilt(ax, f, x0))
    plt.show()


def _numbers(path, who):
    ok = isinstance(path, (list, tuple)) and len(path) > 0 and all(
        isinstance(p, (int, float, np.integer, np.floating)) and np.isfinite(p) for p in path)
    if not ok:
        raise ValueError("%s has no numbers in it yet. Finish the 🎯 lines of Piece 2 and "
                         "run its check until it says Right, then run this cell again." % who)


def show_climb(f, path):
    """The hill with every place where the grasshopper stood, joined by its jumps."""
    _numbers(path, "The path of the climb")
    jumps = len(path) - 1
    fig, ax = hill_axes(f, "%d jumps, from x = %.2f to x = %.3f" % (jumps, path[0], path[-1]))
    ys = [f(p) for p in path]
    for a, b, fa, fb in zip(path[:-1], path[1:], ys[:-1], ys[1:]):
        if jumps <= 40:
            ax.annotate("", xy=(b, fb), xytext=(a, fa), zorder=4,
                        arrowprops=dict(arrowstyle="-|>", color=TRAIL, lw=1.1, shrinkA=3,
                                        shrinkB=3, mutation_scale=9))
    cmap = LinearSegmentedColormap.from_list("trail", [TRAIL_FAINT, TRAIL])
    ax.scatter(path, ys, c=np.arange(len(path)), cmap=cmap, s=26, zorder=5,
               edgecolor=TRAIL, linewidth=0.6)
    ax.annotate("start", (path[0], ys[0]), xytext=(path[0] - 0.25, ys[0] + 1.3),
                ha="center", color=TRAIL, fontsize=10)
    face = 1 if path[-1] >= path[0] else -1
    draw_hopper(ax, path[-1], ys[-1], face=face, tilt=slope_tilt(ax, f, path[-1]))
    plt.show()


def show_trace(f, path, alpha, max_jumps):
    """Where the grasshopper stands after each jump, next to the climb with alpha = 0.1."""
    _numbers(path, "The path of the climb")
    fig, ax = plt.subplots(figsize=(7.2, 3.2))
    fig.subplots_adjust(left=0.1, right=0.985, bottom=0.18, top=0.88)
    ax.set_facecolor(SKY)
    if max(abs(v) for v in path) <= 50:               # a run that stays near the hill
        reference = gradient_ascent(f, path[0])[1]
        ax.plot(range(len(reference)), reference, color=TRAIL_FAINT, lw=2.2, label="alpha = 0.1")
    ax.plot(range(len(path)), path, color=TRAIL, lw=1.4, marker="o" if len(path) <= 60 else None,
            ms=3.5, label="alpha = %g" % alpha)
    ax.axhline(X_TOP, color=PEAK, lw=1.2, linestyle="--", label="the top (x = 3)")
    ax.set_xlabel("jump number", color=INK)
    ax.set_ylabel("place x after the jump", color=INK)
    ax.set_title("where the grasshopper stands, jump by jump", loc="left", fontsize=11, color=INK)
    ax.tick_params(colors=INK)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=9, loc="best")
    plt.show()
    jumps, x_end = len(path) - 1, path[-1]
    if jumps < max_jumps:
        print("alpha = %g: the grasshopper settled at x = %.3f after %d jumps." % (alpha, x_end, jumps))
    elif X_MIN <= x_end <= X_MAX:
        print("alpha = %g: after %d jumps the safety net stopped the grasshopper. It is still on its "
              "way, at x = %.3f." % (alpha, jumps, x_end))
    else:
        print("alpha = %g: every jump overshoots the top by more than the one before, so the "
              "grasshopper is flung off the hill. After %d jumps the safety net stopped it, at "
              "x = %.1f." % (alpha, jumps, x_end))


# ==== checks =================================================================
# A check prints "Right." or "Not yet." with a hint, and writes its verdict in PIECE_DONE.

BLANK_1 = ("Piece 1 is still blank. Replace every `...` on the lines marked 🎯 1a and 1b, run that "
           "cell, then run this check again.")
BLANK_2 = ("Piece 2 is still blank. Replace every `...` on the lines marked 🎯 2a, 2b and 2c, run "
           "that cell, then run this check again.")
NO_HILL = "Run the cell of section 1 first. It defines the hill f and its exact slope df."


def _verdict(piece, ok, message):
    PIECE_DONE[piece] = ok
    text = ("✅ Right. " if ok else "❌ Not yet. ") + message
    print(textwrap.fill(text, width=100, subsequent_indent="   "))


def _is_number(v):
    return isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool)


def _blank_error(err):
    return "ellipsis" in str(err).lower()


def _has_blank(function):
    """True if the code of the function still has the placeholder `...` in it."""
    try:
        tree = ast.parse(textwrap.dedent(inspect.getsource(function)))
    except (OSError, TypeError, SyntaxError):
        return False                                   # no source to read: the checks look at behaviour
    return any(isinstance(node, ast.Constant) and node.value is Ellipsis for node in ast.walk(tree))


def check_grad(grad_function):
    """Compare grad with the exact slope df at four places, and on a different hill."""
    if "f" not in globals() or "df" not in globals():
        return _verdict("grad", False, NO_HILL)
    hill, exact = globals()["f"], globals()["df"]
    if _has_blank(grad_function):
        return _verdict("grad", False, BLANK_1)
    places = [0.5, 2.0, 3.0, 5.0]
    try:
        mine = [grad_function(hill, x) for x in places]
    except Exception as err:
        if _blank_error(err):
            return _verdict("grad", False, BLANK_1)
        return _verdict("grad", False, "Your grad stopped with an error: %s: %s. Read the line that "
                        "the error points to." % (type(err).__name__, err))
    if any(v is Ellipsis for v in mine):
        return _verdict("grad", False, BLANK_1)
    if not all(_is_number(v) for v in mine):
        return _verdict("grad", False, "grad(f, x) has to return a number, and it returned %r. Does "
                        "the cell end with `return grad`?" % (mine[0],))
    want = [exact(x) for x in places]
    print("      x    your grad    exact slope df")
    for x, m, w in zip(places, mine, want):
        print("  %5.1f  %10.3f  %14.3f" % (x, m, w))
    if max(abs(m - w) for m, w in zip(mine, want)) > 0.05:
        if all(abs(m + w) < 0.05 for m, w in zip(mine, want)):
            why = ("Every sign is the wrong way round: where the hill goes up, your numbers say it "
                   "goes down. In delta, the altitude after the step comes first.")
        elif all(abs(m / w - 0.01) < 0.002 for m, w in zip(mine, want) if abs(w) > 1):
            why = "Your numbers are about 100 times too small. Was the change in altitude divided by eps?"
        elif all(abs(m / w - 2) < 0.05 for m, w in zip(mine, want) if abs(w) > 1):
            why = ("Your numbers are about twice the exact slope. Check the width of the step that "
                   "delta covers, and divide by that width.")
        elif all(abs(m) > 20 * (abs(w) + 1) for m, w in zip(mine, want)):
            why = ("Your numbers are far too big. delta is the altitude after a step of size eps, "
                   "minus the altitude here.")
        else:
            why = "At least one of your numbers is more than 0.05 away from the exact slope."
        return _verdict("grad", False, why)
    ramp = grad_function(lambda x: 3 * x + 1, 0.7)
    if not _is_number(ramp) or abs(ramp - 3) > 1e-6:
        return _verdict("grad", False, "Your grad is right on this hill and wrong on a different one. "
                        "On a ramp that rises 3 for every step to the right it should give 3, and it "
                        "gives %s. Does grad use only the function f that it receives?" % (ramp,))
    return _verdict("grad", True, "Your gradient is within 0.01 of the exact slope at every place, and "
                    "it also works on a different hill. The small difference is the price of "
                    "measuring over a step of 0.01.")


def check_ascent(ascent_function):
    """Climb from the left slope and from the right slope, and test that alpha matters."""
    if "f" not in globals() or "grad" not in globals():
        return _verdict("ascent", False, NO_HILL)

    def fail(message):
        if not PIECE_DONE["grad"]:
            message += " The check of Piece 1 has not said Right yet, so look there too."
        _verdict("ascent", False, message)

    hill, grad_function = globals()["f"], globals()["grad"]
    if _has_blank(ascent_function):
        if _has_blank(grad_function):
            return _verdict("ascent", False, "Piece 2 builds on Piece 1, and grad is still blank. "
                            "Finish Piece 1 first and get a Right from its check.")
        return fail(BLANK_2)
    try:
        first = grad_function(hill, 0.5)
    except Exception:
        first = None
    if not _is_number(first):
        return _verdict("ascent", False, "Piece 2 builds on Piece 1, and grad does not give a number "
                        "yet. Finish Piece 1 first and get a Right from its check.")
    runs = {}
    for start, side in [(0.5, "left"), (5.5, "right")]:
        try:
            result = ascent_function(hill, start)
        except Exception as err:
            return fail(BLANK_2 if _blank_error(err) else
                        "Your gradient_ascent stopped with an error: %s: %s. Read the line that the "
                        "error points to." % (type(err).__name__, err))
        if not (isinstance(result, tuple) and len(result) == 2 and isinstance(result[1], list)):
            return fail("gradient_ascent has to end with `return x, path`. That line is given, so "
                        "check that it is still there.")
        x_end, path = result
        if x_end is Ellipsis or any(p is Ellipsis for p in path):
            return fail(BLANK_2)
        if not _is_number(x_end):
            return fail("gradient_ascent returned %r as the final place, which is not a number." % (x_end,))
        runs[side] = (start, x_end, path)
    for side, (start, x_end, path) in runs.items():
        jumps = len(path) - 1
        if jumps == 0:
            if side == "left":
                return fail("From x = 0.5 the grasshopper did not jump at all, so the loop condition "
                            "(🎯 2a) was already False. It has to be True while the ground is not flat.")
            return fail("From x = 5.5, on the right slope, the grasshopper did not jump at all. There "
                        "the gradient is negative. The loop condition (🎯 2a) has to look at the size "
                        "of g, and abs(g) gives it.")
        if jumps >= 1000:
            return fail("After 1000 jumps the grasshopper was still jumping, and the safety net stopped "
                        "it. Look at 🎯 2c (measure the gradient again, at the new place) and at 🎯 2b "
                        "(the jump).")
        if (x_end - start) * (X_TOP - start) <= 0:
            return fail("From x = %.1f the grasshopper walked downhill and ended at x = %.3g. The jump "
                        "has to follow the gradient: x grows when g is positive and shrinks when g is "
                        "negative." % (start, x_end))
        if abs(x_end - X_TOP) > 0.1:
            return fail("From x = %.1f the grasshopper stopped at x = %.3f, which is %.2f away from the "
                        "top at x = %g. Is the loop condition (🎯 2a) strict enough? The slide stops "
                        "when g is 0.01 or less." % (start, x_end, abs(x_end - X_TOP), X_TOP))
    jumps_default = len(runs["left"][2]) - 1
    jumps_big = len(ascent_function(hill, 0.5, alpha=0.5)[1]) - 1
    if jumps_big >= jumps_default:
        return fail("With alpha = 0.5 the grasshopper needed %d jumps and with alpha = 0.1 it needed "
                    "%d. A bigger alpha means longer jumps, so alpha has to appear in 🎯 2b."
                    % (jumps_big, jumps_default))
    start, x_end, path = runs["left"]
    other = runs["right"]
    return _verdict("ascent", True, "From x = %.1f the grasshopper climbed to x = %.3f in %d jumps, "
                    "within %.3f of the top. From x = %.1f, on the other slope, it jumped to the left "
                    "and stopped at x = %.3f." % (start, x_end, jumps_default, abs(x_end - X_TOP),
                                                    other[0], other[1]))


print("Ready. Nothing was installed and nothing was downloaded.")

## 1. The hill

The hill is a function `f`. Give it a place `x` and it returns the altitude at that place. Ours
is an arch that stands on the ground between x = 0 and x = 6. Its top is at x = 3, where the
altitude is 9.

`df` is the exact slope of the hill, the kind of thing you compute in a calculus class. The
grasshopper does not know it and your code will never use it. The check cells use `df` to find
out whether your gradient is right.

In [ ]:
def f(x):
    return x * (6 - x)       # the altitude at the place x: an arch with its top at x = 3

def df(x):
    return 6 - 2 * x         # the exact slope of f (only the check cells use it)

x0 = 0.5                     # where the grasshopper starts, on the left slope

draw_hill(f, x0)             # draws the hill and the grasshopper

## 2. 🎯 Piece 1: the gradient

The grasshopper cannot read the slope off a formula. It measures the slope by trying a small
step to the right and looking at how much the altitude changes. In pseudocode:

```
grad(f, x):
    eps = a small step (0.01)
    delta = the altitude after the step, minus the altitude here
    grad = delta divided by eps
    return grad
```

`f` is the function (the hill) and `x` is the location (where the grasshopper stands). Write the
two lines marked 🎯, run the cell, and then run the check below it.

<details><summary>💡 Hint</summary>

The altitude here is `f(x)`. The altitude after a step of size `eps` to the right is
`f(x + eps)`.
</details>

In [ ]:
def grad(f, x):
    eps = 0.01                  # a small step: how far the grasshopper tries
    delta = ...                 # 🎯 1a) the altitude after the step, minus the altitude here
    grad = ...                  # 🎯 1b) the change in altitude divided by the step size
    return grad

In [ ]:
check_grad(grad)

## 3. 🎯 Piece 2: gradient ascent

Now the climb. The grasshopper measures the gradient `g` where it stands. While the ground is
not flat, it jumps by `alpha * g` and measures again. A steep slope gives a long jump and a
gentle slope gives a short one. A negative gradient gives a jump to the left, which is the
uphill direction there. In pseudocode:

```
gradient_ascent(f, x, alpha):
    g = the gradient where the grasshopper stands
    while the ground is not flat (g is not close to 0, the limit is 0.01):
        jump: move x by alpha times g
        measure the gradient again, at the new place
    return where the grasshopper ended up
```

`alpha` says how far the grasshopper jumps for each unit of gradient. It is a different small
number from `eps`. The number `eps` is how far the grasshopper tries a step to measure the
slope, and `alpha` is how far it jumps after measuring.

Three lines are yours, the ones marked 🎯. The other lines are given. They remember the places
where the grasshopper has stood, so that a later cell can draw them, and they add a safety net:
a grasshopper that has not stopped after `max_jumps` jumps gives up. Write the three lines, run
the cell, and then run the check below it.

<details><summary>💡 Hint</summary>

The gradient can be negative, on the right slope of the hill. Look at the size of `g`, which is
`abs(g)`. The jump is `alpha * g`, added to `x`. After the jump the grasshopper stands at a new
place, so `g` has to be measured again with `grad`.
</details>

In [ ]:
def gradient_ascent(f, x, alpha=0.1, max_jumps=1000):
    path = [x]                       # given: the places where the grasshopper has stood
    g = grad(f, x)                   # given: the gradient where it starts
    while ...:                       # 🎯 2a) keep jumping while the ground is not flat
        x = ...                      # 🎯 2b) jump: move by alpha times the gradient
        g = ...                      # 🎯 2c) measure the gradient again, at the new place
        path.append(x)               # given: remember the new place
        if len(path) > max_jumps:    # given: the safety net, give up after max_jumps jumps
            break
    return x, path                   # given: where it ended up, and the places it visited

In [ ]:
check_ascent(gradient_ascent)

## 4. Run it and look

Now watch the climb. Think first: as the grasshopper nears the top, will its jumps get longer or
shorter? Then run the cell. Every dot is a place where the grasshopper stood and every arrow is
one jump.

In [ ]:
x_end, path = gradient_ascent(f, x0)       # climb from x0 with alpha = 0.1
show_climb(f, path)                        # draw the places where the grasshopper stood

print("number of jumps :", len(path) - 1)
print("final place x   :", round(x_end, 3))
print("final altitude  :", round(f(x_end), 3))
print("the top of the hill is at x = 3, with altitude", f(3))

The jumps get shorter as the grasshopper nears the top. A jump is `alpha * g` and the gradient `g`
shrinks on the way up. The grasshopper stops as soon as the gradient is within 0.01 of zero,
which happens a little before the exact top at x = 3.

## 5. Play with alpha

The number `alpha` decides how far the grasshopper jumps. There is nothing to fill in here.
Change `ALPHA` in the next cell and run it again to see what happens. The picture shows where
the grasshopper stands after each jump, and the pale line is the climb with `alpha = 0.1`, for
comparison.

First a very small `alpha`.

In [ ]:
ALPHA = 0.005      # @param {type:"number"}
MAX_JUMPS = 1000   # the safety net: the grasshopper gives up after this many jumps

x_end, path = gradient_ascent(f, x0, alpha=ALPHA, max_jumps=MAX_JUMPS)
show_trace(f, path, ALPHA, MAX_JUMPS)

Now a very large `alpha`. Think first: what happens when a jump is longer than the distance to
the top? The safety net is set to 30 jumps here, so that the cell ends even if the grasshopper
never settles.

In [ ]:
ALPHA = 1.1        # @param {type:"number"}
MAX_JUMPS = 30     # the safety net: the grasshopper gives up after this many jumps

x_end, path = gradient_ascent(f, x0, alpha=ALPHA, max_jumps=MAX_JUMPS)
show_trace(f, path, ALPHA, MAX_JUMPS)

## 6. What you built

- `grad(f, x)` measures the slope of a hill by trying a small step.
- `gradient_ascent(f, x, alpha)` jumps in the direction of the gradient until the ground is flat.
- A small `alpha` gives many tiny jumps. A large `alpha` overshoots the top, and on this hill an
  `alpha` of 1.1 flings the grasshopper off.

Both functions work for any hill `f`. In the next part of the morning, the hill is the average
number of coins that Mara earns in a day.